# DPDP private Colab worker

Before the first run, add `GITHUB_TOKEN` and `DPDP_WORKER_TOKEN` in Colab's **Secrets** panel (key icon) and enable notebook access. Then choose **Runtime → Run all**.

In [ ]:
import base64, os, pathlib, subprocess
from google.colab import userdata

repo = pathlib.Path('/content/DPDP-Benchmark')
github_token = userdata.get('GITHUB_TOKEN')
worker_token = userdata.get('DPDP_WORKER_TOKEN')
if not github_token or not worker_token:
    raise RuntimeError('Add GITHUB_TOKEN and DPDP_WORKER_TOKEN in Colab Secrets and enable notebook access.')

if not (repo / '.git').is_dir():
    auth = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
    subprocess.run([
        'git', '-c', f'http.extraHeader=Authorization: Basic {auth}',
        'clone', 'https://github.com/Hari-Pi/DPDP-Benchmark.git', str(repo),
    ], check=True)
else:
    auth = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
    subprocess.run([
        'git', '-C', str(repo), '-c', f'http.extraHeader=Authorization: Basic {auth}',
        'pull', '--ff-only',
    ], check=True)

env = os.environ.copy()
env['DPDP_WORKER_TOKEN'] = worker_token
subprocess.run(['bash', 'scripts/start_colab_worker.sh'], cwd=repo, env=env, check=True)